### Electrostatics: Poisson Equation for a Point Charge with Neumann and Dirichlet Boundary Conditions

**Numerical solution of the Poisson equation.** In this example the potential
$\varphi$ of a proton is computed numerically. Since the analytical solution of
the Poisson problem

$$\operatorname{div}(\varepsilon \operatorname{grad} \varphi) = -\rho$$

is known, the numerical result can be verified against it.

The advantage of the Finite Integration Technique is that the Poisson equation
translates directly into the discrete setting:

$$\tilde{\mathbf{S}}\, \mathbf{M}_\varepsilon\, \mathbf{G}\, \Phi = -q .$$

Each factor has a clear meaning on the staggered grid:

- $\Phi \in \mathbb{R}^{N_p}$ collects the electric potential at the $N_p$ primal
  nodes.
- $\mathbf{G} \in \mathbb{R}^{3N_p \times N_p}$ is the discrete gradient. It maps
  nodal potentials to the voltages along the primal edges, $\hat{e} = -\mathbf{G}\Phi$,
  by taking the difference between the two nodes bounding each edge. The factor
  three counts the $u$-, $v$- and $w$-directed edges.
- $\mathbf{M}_\varepsilon \in \mathbb{R}^{3N_p \times 3N_p}$ is the permittivity
  material matrix, the only place where geometry and material enter. It is
  diagonal, each entry being the dual facet area divided by the primal edge
  length, weighted by the permittivities of the surrounding cells. It converts
  edge voltages into the electric fluxes through the corresponding dual facets,
  $\hat{\hat{d}} = \mathbf{M}_\varepsilon \hat{e}$.
- $\tilde{\mathbf{S}} \in \mathbb{R}^{N_p \times 3N_p}$ is the discrete divergence
  on the dual grid. It sums the fluxes through the six facets bounding each dual
  volume, giving the enclosed charge — Gauss's law, $\tilde{\mathbf{S}}\hat{\hat{d}} = q$,
  exactly and without discretisation error.
- $q \in \mathbb{R}^{N_p}$ holds the charge contained in each dual volume. Because
  it is already a volume-integrated charge density, it carries units of coulombs
  and can be assigned directly: placing a proton means writing $1.602 \times 10^{-19}$
  into the single entry found. No division by a cell volume is needed.

Because $\tilde{\mathbf{S}} = -\mathbf{G}^\mathsf{T}$ holds exactly on the
staggered grid, the system can be rewritten as

$$\mathbf{G}^\mathsf{T} \mathbf{M}_\varepsilon\, \mathbf{G}\, \Phi = q .$$

**The problem.** Although the discrete Poisson system is assembled easily in FIT
notation, two points need to be kept in mind.

When the gradient operator acts on a scalar nodal potential, it returns an edge
quantity. In each direction the domain holds $N$ nodes but only $N-1$ edges, so
the last plane of entries in the state vector corresponds to edges that do not
exist:

```
        node 1     node 2     node 3     node 4     node 5
          ●━━━━━━━━━━●━━━━━━━━━━●━━━━━━━━━━●━━━━━━━━━━●- - - - - ⇢
           edge 1     edge 2     edge 3     edge 4     edge 5
                                                        (dead)

        N = 5 nodes   →   4 real edges   +   1 dead edge
```

These *dead* — or *ghost* — edges must be removed before the system is solved;
`get_ghost_matrix` builds the projection that does so.

Once that is done, the Poisson problem comes with homogeneous Neumann boundary
conditions naturally. This notebook discusses the two basic cases, homogeneous
Neumann and Dirichlet boundary conditions, along with the difficulties each
brings.

In [ ]:
import Pkg
# This activates the Project.toml located two folders up from this script
Pkg.activate(joinpath(@__DIR__, "..", "..")) 
Pkg.instantiate() 
using FITToolbox;
using CairoMakie;
using LinearAlgebra; 

**Defining the domain.** The grid spans $100\;\text{m}$ in each direction at a
resolution of $2\;\text{m}$. In the current version of FITToolbox the origin of
the domain is always $[0, 0, 0]$.

Each direction may be given its own resolution. The toolbox works in metres by
default. The background material properties are set when the domain is created;
here they are the vacuum values for conductivity $\sigma$, relative permittivity
$\varepsilon_r$ and relative permeability $\mu_r$.

For a non-equidistant grid, pass the edge lengths directly:

```julia
create_domain(Edges_X, Edges_Y, Edges_Z; units="m", σ=0.0, ε_r=1.0, μ_r=1.0)
```

Each vector holds the individual cell widths along that direction, so the spacing
may vary from cell to cell. Fifty edges of $2\;\text{m}$ each reproduce the
equidistant domain above ($50$ cells, $51$ nodes per direction):

```julia
Edges_X = fill(2.0, 50)
Edges_Y = fill(2.0, 50)
Edges_Z = fill(2.0, 50)
```

In [ ]:
x_size, y_size, z_size = 100,100,100
resolution = 2

MyDomain = create_domain([x_size,y_size,z_size],[resolution,resolution,resolution];units="m",σ=0,ε_r=1,μ_r=1); 

**Placing the charge.** A single proton is placed at the centre of the domain. In
the Finite Integration Technique the charge is a volume-integrated charge density,
so it is associated with a dual volume rather than with a point. The corresponding
index into the state vector is found with `get_index_entity`:

```julia
p_q = get_index_entity(MyDomain, DualVolume(), x_size/2.0, y_size/2.0, z_size/2.0;
                       units="m", atol=1e-9)
```

Each dual volume surrounds one primal node, so `p_q` is an index in $1 \dots N_p$.
The `atol` argument sets how far the requested position may lie from the nearest
dual volume centre before a warning is issued.

In [ ]:
p_q = get_index_entity(MyDomain, DualVolume(), x_size/2.0, y_size/2.0, z_size/2.0; units="m", atol=1e-9)

q = zeros(Float64, MyDomain.Np);
q[p_q] = 1.602e-19; #C

In [ ]:
# The FIT permittivity matrix. Diagonal, with each entry the dual facet area
# divided by the primal edge length, weighted by the permittivities of the
# surrounding cells. Absolute, i.e. it already contains ε₀.
Mε = get_permittivity(MyDomain)

# Diagonal projection matrix: 1 on every real edge, 0 on the dead ones.
# Applied from the left, it zeroes the rows of G belonging to dead edges.
dead_edges = get_ghost_matrix(MyDomain)

G = dead_edges * get_gradient(MyDomain, Primal())

# System matrix. Dropping the dead edges leaves homogeneous Neumann conditions
# on all six faces — no flux crosses the boundary. Symmetric and positive
# semi-definite, but singular with a one-dimensional null space: the grid stays
# connected, so Φ is determined only up to an additive constant.
L = G' * Mε * G

In [ ]:
# With homogeneous Neumann conditions alone the system is singular: Φ is
# determined only up to an additive constant. Pinning one node fixes the gauge.
#
# R keeps the free nodes: 1 everywhere except the pinned node.
pinned = zeros(Float64, MyDomain.Np)
pinned[1] = 1.0
R = Diagonal(1.0 .- pinned)

# D is the complement, 1 on the pinned node. It supplies the diagonal entry for
# the row R zeroes out, which is what keeps Lbc invertible.
D = Diagonal(pinned);

**Neutralising the right-hand side.** Homogeneous Neumann conditions allow no
flux through the boundary, so Gauss's law requires the enclosed charge to be
zero. A single proton violates this: $\mathbf{1}^\mathsf{T}q \neq 0$ places the
right-hand side outside the range of $\mathbf{L}$, and the system becomes
inconsistent. Note that this is a separate issue from the singularity of
$\mathbf{L}$ — pinning a node fixes the gauge, but an inconsistent system has no
solution to pin down.

Subtracting the mean removes the component of $q$ along the null space:

$$q \;\leftarrow\; q - \frac{\mathbf{1}^\mathsf{T}q}{N_p}\,\mathbf{1}$$

Physically this distributes a uniform background charge of equal magnitude and
opposite sign across all $N_p$ nodes, leaving the domain neutral. The problem
being solved is therefore a proton embedded in a neutralising background rather
than an isolated one, and the comparison against $1/(4\pi\varepsilon r)$ below
will drift at larger radii for that reason.

Both corrections are needed, and they do different things. Pinning a node is not
purely a gauge choice: the pinned equation is removed from the system, so the
remaining $N_p - 1$ rows still enforce Gauss's law and telescope to a boundary
flux that the Neumann condition sets to zero. Any residual charge imbalance has
nowhere to go but the deleted equation, and the pinned node becomes a point sink
absorbing the entire flux $-q$ — giving a dipole-like field between the proton at
the centre and a sink in the corner. Neutralising the right-hand side first
leaves nothing to absorb, so the pinning reduces to a pure gauge fix and the
electric field $\hat{e} = -\mathbf{G}\Phi$ is unaffected by it.

The Dirichlet case needs no such correction. Grounding all six faces lets the
flux leave the domain entirely: the boundary carries an induced surface charge
of $-q$, so the full flux emitted by the proton terminates there and Gauss's law
is satisfied without modifying the source. The compensating charge sits on the
boundary, where it belongs, instead of being smeared through the volume.

In [ ]:
rhs = copy(q);
rhs .-= sum(rhs)/MyDomain.Np;

In [ ]:
# Impose the pinned node. R*L*R' zeroes row and column 1 of L, keeping the system
# symmetric; adding D puts a 1 on the now-empty diagonal entry, so row 1 reads
# Φ₁ = rhsbc₁. Zeroing the column as well as the row is what preserves symmetry —
# a plain row replacement would not.
Lbc = D + R * L * R'

# Matching RHS: R zeroes entry 1, so the pinned node is set to 0 V. A non-zero
# value would go here rather than in Lbc.
rhsbc = R * rhs

# Lbc is now positive definite — L had a one-dimensional null space and pinning
# one node removes it — so a Cholesky factorisation applies. check=true raises
# PosDefException rather than returning a silently wrong factor.
Φ_Neumann = cholesky(Lbc; check=true) \ rhsbc;

In [ ]:
# plot_nodal_values takes a nodal quantity — one value per primal node, length Np
# — and draws a two-dimensional cut through it. It interpolates the data onto the
# slice, renders it as a heatmap, and optionally overlays the streamlines of the
# negative gradient. Primal() states that the data lives on the primal grid.
#
# Z() selects the slice orientation: the cut plane has its normal along z, so the
# x–y plane is shown, and pos gives where along z the cut is taken — here the
# domain centre, level with the proton. X() and Y() give the other two
# orientations, each taking pos along its own axis.
#
# The heatmap shows Φ on a log colour scale (clip_range sets the decades, since
# the auto-range would hit the zero from the pinned node); with
# plot_negative_gradient the white streamlines trace the in-plane components of
# ē = -GΦ, computed internally from the same potential.
#
# Near the proton the field is radial, as expected. Toward the boundary the lines
# turn to run parallel to the faces — the Neumann condition forbids any normal
# component — and converge on the corners, where the tangential directions of
# adjoining faces meet. This is geometry, not a sink: the lines terminate on the
# neutralising background charge distributed across the whole volume.
plot_nodal_values(MyDomain, Primal(), Φ_Neumann, Z();
                  pos = z_size/2.0, units="m",
                  clip_range = (1e-12, 1e-8),
                  logscale = true,
                  plot_negative_gradient = true)

**Homogeneous Dirichlet boundary conditions.** The second standard case grounds
the outer surface of the domain, $\Phi = 0$ on all six faces. Physically this
models the proton sitting inside a closed, perfectly conducting box held at zero
potential.

This removes both difficulties of the Neumann case at once. The grounded surface
is free to carry an induced charge, so the flux emitted by the proton terminates
there: the boundary accumulates exactly $-q$, Gauss's law is satisfied over the
whole domain, and the source can be used unmodified. No neutralising background
is needed, and the potential is fixed absolutely rather than up to a constant —
there is no gauge freedom left to eliminate.

The price is that the box is now part of the model. The image charge on the walls
produces a field of its own, so the solution is not that of an isolated proton in
free space. Near the centre the difference is small; approaching the boundary the
computed potential is forced to zero, whereas the analytical $1/(4\pi\varepsilon r)$
decays but stays positive. The comparison below therefore agrees well at small
radii and diverges near the walls.

**Imposing the condition.** The boundary nodes are identified by listing the six
faces and asking for the corresponding nodal projection. `NodalComponent()`
selects an $N_p \times N_p$ matrix, as opposed to the $3N_p \times 3N_p$ form
used for edge quantities:

$$\mathbf{R}_{pp} = \begin{cases} 1 & \text{$p$ interior} \\ 0 & \text{$p$ on the boundary}\end{cases}
\qquad \mathbf{D} = \mathbf{I} - \mathbf{R}$$

The constrained system is then assembled as

$$\mathbf{L}_{bc} = \mathbf{D} + \mathbf{R}\,\mathbf{L}\,\mathbf{R}, \qquad
  q_{bc} = \mathbf{R}\,q .$$

Multiplying by $\mathbf{R}$ from both sides zeroes the boundary rows *and*
columns, which keeps the matrix symmetric — replacing rows alone would not.
Adding $\mathbf{D}$ then places a one on each emptied diagonal entry, so every
boundary row reduces to $\Phi_p = 0$. A non-zero boundary potential would be
imposed by adding $\mathbf{D}\Phi_{bc}$ to the right-hand side rather than by
changing $\mathbf{L}_{bc}$.

Constraining the whole shell removes the null space entirely, so $\mathbf{L}_{bc}$
is symmetric positive definite and a Cholesky factorisation applies.

In [ ]:
# All six outer faces of the domain. Each entry names a direction and which side
# of it, so this list covers the complete boundary shell.
outer_faces = [
    (X(), Positive()), (X(), Negative()),
    (Y(), Positive()), (Y(), Negative()),
    (Z(), Positive()), (Z(), Negative())
]

# Same system matrix as before: the ghost matrix removes the dead edges, and
# L = GᵀM_εG is symmetric positive semi-definite with a one-dimensional null
# space. The boundary conditions below are what make it definite.
G = get_ghost_matrix(MyDomain) * get_gradient(MyDomain, Primal())
L = G' * get_permittivity(MyDomain) * G

# Nodal projection matrices. NodalComponent() gives an Np×Np matrix rather than
# the 3Np×3Np used for edge quantities.
# R keeps the interior: 1 on interior nodes, 0 on the boundary shell.
R = get_boundary_matrix(MyDomain, outer_faces, NodalComponent())

# D is the complement, 1 on the boundary shell. It supplies the diagonal entries
# for the rows R zeroes out, which is what keeps Lbc invertible.
D = Diagonal(ones(MyDomain.Np)) - R

# R*L*R zeroes the boundary rows and columns, preserving symmetry; D fills the
# empty diagonal so each boundary row reads Φ = 0, i.e. grounded.
# Unlike the Neumann case, q is used unmodified: the grounded boundary carries an
# induced surface charge of -q, so the full flux terminates there and Gauss's law
# is satisfied without a compensating background.
Lbc = D + R * L * R
rhsbc = R * q

# Pinning an entire shell removes the null space, so Lbc is positive definite and
# Cholesky applies. check=true raises PosDefException rather than returning a
# silently wrong factor.
Φ_Dirichlet = cholesky(Lbc; check=true) \ rhsbc;

#ē = -G * Φ;

In [ ]:
# The heatmap shows Φ on a log colour scale (clip_range sets the decades, since
# the auto-range would hit the zero from the pinned node); the white streamlines
# trace the in-plane components of ē = -GΦ.
#
# The field meets the walls perpendicularly: Φ = 0 along the surface leaves no
# tangential component, so every line terminates on the induced surface charge
# rather than turning to run along the face. Compare with the Neumann plot above,
# where the opposite holds — no flux crosses the boundary, so the lines are
# forced parallel to it.
plot_nodal_values(MyDomain, Primal(), Φ_Dirichlet, Z();
                  pos = z_size/2.0, units="m",
                  clip_range = (1e-12, 1e-8),
                  logscale = true,
                  plot_negative_gradient = true)

**Comparison against the analytical solution.** The potential of a point charge in
unbounded space is known exactly,

$$\varphi(r) = \frac{q}{4\pi\varepsilon_0 r},$$

so the numerical results can be checked directly against it. The cell below
evaluates this expression at every node, then samples all three solutions along a
line running outward from the centre and plots them on a logarithmic scale.

The sampling line follows the $x$–$y$ diagonal of the centre $z$-plane rather
than a coordinate axis. The diagonal reaches $\sqrt{2}\cdot 50 \approx 71\;\text{m}$
before leaving the domain, so it covers a wider range of radii than any axis-aligned
cut. The analytical expression is softened as $\max(r, 0.05)$ to avoid the
singularity at $r = 0$; the sampled line never comes close to that node, so the
softening has no effect on the comparison.

**Interpreting the deviation.** Out to roughly $10\;\text{m}$ all three curves
coincide. This is the result worth noting: away from the boundary, FIT reproduces
the $1/r$ decay accurately, even though the charge occupies a single dual volume
and is therefore about as harsh a source as the discretisation can be given.

Beyond that both numerical curves fall below the analytical one. Neither is in
error — they answer a different question. The analytical curve describes an
isolated charge in free space, whereas both computations place it inside a
$100\;\text{m}$ box, and in each case the box contributes a negative correction:
the grounded walls carry an induced charge of $-q$, and the Neumann background
carries $-q$ smeared through the volume. Either way the far field is suppressed
relative to free space. The deviation is thus a statement about the domain size,
not about the discretisation, and it shrinks as the box is enlarged.

The two boundary conditions differ in *how* the decay proceeds. The Dirichlet
curve drops steeply past $50\;\text{m}$ and leaves the plot: $\Phi = 0$ is imposed
on the wall, so on a logarithmic axis the curve falls towards $-\infty$ as it
approaches. The Neumann curve instead flattens onto a floor near $10^{-11.5}\;\text{V}$
— no flux may cross the boundary, so the potential cannot be driven to zero there,
and it settles at the level set by the neutralising background together with the
constant fixed by the pinned node.

In [ ]:
# The charge sits at the domain centre; measure everything from there.
centre = (x_size/2, y_size/2, z_size/2)
node(i, j, k) = 1 + (i-1) + (j-1)*Nu + (k-1)*Nu*Nv
radius(i, j, k) = hypot(centre[1] - MyDomain.nodes_u[i],
                        centre[2] - MyDomain.nodes_v[j],
                        centre[3] - MyDomain.nodes_w[k])

Nu, Nv, Nw, Np = MyDomain.Nu, MyDomain.Nv, MyDomain.Nw, MyDomain.Np
i_c = findlast(x -> x <= centre[1], MyDomain.nodes_u)
k_c = findlast(x -> x <= centre[3], MyDomain.nodes_w)

# Analytical free-space potential at every node. max(r, 0.05) softens the
# singularity at r = 0, which the sampled line never reaches anyway.
Φ_analytical = [1.602e-19 / (4π * 8.8541878188e-12 * max(radius(i,j,k), 0.05))
                for k = 1:Nw, j = 1:Nv, i = 1:Nu] |> a -> vec(permutedims(a, (3,2,1)))

# Sample along the u–v diagonal of the centre z-plane, running from the corner
# inward. The diagonal reaches sqrt(2)·50 ≈ 71 m, further than any axis.
line = [node(i, i, k_c) for i = i_c-1:-1:1]
r    = [radius(i, i, k_c) for i = i_c-1:-1:1]

fig = Figure(size = (800, 600))
ax = Axis(fig[1, 1], xlabel = "Distance from Center (m)", ylabel = "Potential Φ (V)",
          title = "Potential Comparison on Grid Nodes", yscale = log10)
lines!(ax, r, Φ_analytical[line]; label = "Analytical", linestyle = :dash, color = :black, linewidth = 2)
lines!(ax, r, Φ_Neumann[line];    label = "Neumann",   color = :blue, linewidth = 2)
lines!(ax, r, Φ_Dirichlet[line];  label = "Dirichlet", color = :red,  linewidth = 2)
axislegend(ax, position = :rt)
fig

**Imposing the analytical values on the boundary.** The deviation seen above comes
from the finite domain, not from the discretisation. This can be demonstrated by
replacing the grounded boundary with the analytical potential itself:

$$\Phi_p = \frac{q}{4\pi\varepsilon_0 r_p} \qquad \text{for every boundary node } p .$$

The box then no longer truncates the problem — the boundary carries exactly the
values that an unbounded domain would produce there — and the computed interior
potential follows the analytical curve across the whole domain, not just near the
centre.

What remains is the discretisation error alone. The agreement is therefore close
but not exact: sampling $1/r$ on a grid and representing a point charge by a
single dual volume are approximations, and their error is largest in the first few
cells around the source, where the field varies most sharply. Away from the
charge the field is smooth, FIT is second-order accurate, and the two curves
become difficult to distinguish.

Separating the two error sources this way is a standard verification technique: if
the remaining deviation is small and shrinks under grid refinement, the operators
and material matrices are implemented correctly, and any larger discrepancy in the
grounded case can be attributed to the domain size rather than to a bug.